# Financial RAG Challenge — Treasury Bulletin OfficeQA

In [3]:
!pip -q install pandas numpy tqdm sentence-transformers faiss-cpu huggingface_hub


In [4]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

In [5]:
import os, re, json, shutil
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm
from huggingface_hub import list_repo_files, hf_hub_download
from sentence_transformers import SentenceTransformer
import faiss

REPO_ID = "databricks/officeqa"
YEARS = [2022, 2023, 2024, 2025]
K = 5

PROJECT_DIR = Path("/content/financial_rag_officeqa")
DOC_DIR = PROJECT_DIR / "data" / "docs"
RESULTS_DIR = PROJECT_DIR / "results"
for d in (DOC_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

FNAME_RE = re.compile(r"treasury_bulletin_(\d{4})_(\d{2})")


## 1. Loadingthe answer key
Confirmed schema: `uid, question, answer, source_docs, source_files`.

In [6]:
files = list_repo_files(REPO_ID, repo_type="dataset")
print("Total files in dataset repo:", len(files))

csv_candidates = [f for f in files if "officeqa_full" in f.lower() and f.lower().endswith(".csv")]
if not csv_candidates:
    raise FileNotFoundError("Could not find officeqa_full.csv — check dataset access on Hugging Face.")

csv_path = hf_hub_download(repo_id=REPO_ID, filename=csv_candidates[0], repo_type="dataset")
qa_df = pd.read_csv(csv_path)
print(qa_df.shape)
qa_df.head()


Total files in dataset repo: 2105


officeqa_full.csv:   0%|          | 0.00/155k [00:00<?, ?B/s]

(246, 6)


,uid,question,answer,source_docs,source_files,difficulty
0,UID0001,What were the total expenditures (in millions ...,"2,602",https://fraser.stlouisfed.org/title/treasury-b...,treasury_bulletin_1941_01.txt,hard
1,UID0002,What were the total expenditures of the U.S fe...,507,https://fraser.stlouisfed.org/title/treasury-b...,treasury_bulletin_1944_01.txt,easy
2,UID0003,Using specifically only the reported values fo...,"44,463",https://fraser.stlouisfed.org/title/treasury-b...,treasury_bulletin_1954_02.txt,hard
3,UID0004,Using specifically only the reported values fo...,1608.80%,https://fraser.stlouisfed.org/title/treasury-b...,treasury_bulletin_1941_01.txt\r\ntreasury_bull...,hard
4,UID0005,Using specifically only the reported values fo...,39482.03,https://fraser.stlouisfed.org/title/treasury-b...,treasury_bulletin_1941_01.txt\r\ntreasury_bull...,hard


## 2. Filtering questions to the chosen years
Uses `source_files` directly — no guessing which column holds the date.

In [7]:
def gold_files(cell):
    if pd.isna(cell):
        return []
    parts = re.split(r"[,\s]+", str(cell).strip())
    return [Path(p).stem for p in parts if p]

qa_df["gold_source_files"] = qa_df["source_files"].apply(gold_files)
qa_df["gold_years"] = qa_df["gold_source_files"].apply(
    lambda fs: [FNAME_RE.search(f).group(1) for f in fs if FNAME_RE.search(f)]
)

years_str = [str(y) for y in YEARS]
mask = qa_df["gold_years"].apply(lambda ys: any(y in years_str for y in ys))
qa_recent = qa_df[mask].reset_index(drop=True)

print(f"Filtered to {len(qa_recent)} / {len(qa_df)} questions touching years {YEARS}")
qa_recent.head()


Filtered to 8 / 246 questions touching years [2022, 2023, 2024, 2025]


,uid,question,answer,source_docs,source_files,difficulty,gold_source_files,gold_years
0,UID0008,What percent did the Employment and General Re...,73,https://fraser.stlouisfed.org/title/treasury-b...,treasury_bulletin_2012_06.txt\r\ntreasury_bull...,easy,"[treasury_bulletin_2012_06, treasury_bulletin_...","[2012, 2022]"
1,UID0010,How much does the U.S Treasury have invested i...,935851121560,https://fraser.stlouisfed.org/title/treasury-b...,treasury_bulletin_2025_06.txt,hard,[treasury_bulletin_2025_06],[2025]
2,UID0078,Calculate the percentage difference relative t...,19.96%,https://fraser.stlouisfed.org/title/treasury-b...,treasury_bulletin_2010_12.txt\r\ntreasury_bull...,easy,"[treasury_bulletin_2010_12, treasury_bulletin_...","[2010, 2015, 2024]"
3,UID0081,"At year-end for CY 2022, what was the total ou...","$23,918,635",https://fraser.stlouisfed.org/title/treasury-b...,treasury_bulletin_2023_03.txt,easy,[treasury_bulletin_2023_03],[2023]
4,UID0086,What was the absolute QoQ percent change in to...,4.815,https://fraser.stlouisfed.org/title/treasury-b...,treasury_bulletin_2022_12.txt,hard,[treasury_bulletin_2022_12],[2022]


## 3. Downloading" the .txt bulletins for those years

In [8]:
text_candidates = [f for f in files if f.lower().endswith((".txt", ".md", ".markdown"))]
recent_text_files = [f for f in text_candidates if any(str(y) in f for y in YEARS)]
print("Recent text files found:", len(recent_text_files))

downloaded = []
for f in tqdm(recent_text_files):
    local_path = hf_hub_download(repo_id=REPO_ID, filename=f, repo_type="dataset")
    safe_name = Path(f).name
    target = DOC_DIR / safe_name
    shutil.copy(local_path, target)
    downloaded.append(target)

print(f"Copied {len(downloaded)} bulletins into {DOC_DIR}")


Recent text files found: 15


  0%|          | 0/15 [00:00<?, ?it/s]

treasury_bulletins_parsed/transformed/tr(…):   0%|          | 0.00/334k [00:00<?, ?B/s]

  7%|▋         | 1/15 [00:01<00:19,  1.40s/it]

treasury_bulletins_parsed/transformed/tr(…):   0%|          | 0.00/247k [00:00<?, ?B/s]

...

treasury_bulletins_parsed/transformed/tr(…):   0%|          | 0.00/249k [00:00<?, ?B/s]

 93%|█████████▎| 14/15 [00:15<00:00,  1.09it/s]

treasury_bulletins_parsed/transformed/tr(…):   0%|          | 0.00/244k [00:00<?, ?B/s]

100%|██████████| 15/15 [00:16<00:00,  1.11s/it]

Copied 15 bulletins into /content/financial_rag_officeqa/data/docs


## 4. Parse Year/Month metadata + build baseline vs. engineered chunks

In [9]:
def parse_year_month(path: Path):
    m = FNAME_RE.search(path.name)
    if not m:
        return None, None
    return int(m.group(1)), int(m.group(2))

docs = []
for p in sorted(DOC_DIR.glob("*.txt")):
    year, month = parse_year_month(p)
    text = p.read_text(errors="ignore")
    docs.append({"source_file": p.stem, "year": year, "month": month, "text": text})
docs_df = pd.DataFrame(docs)
print(f"Loaded {len(docs_df)} documents")


Loaded 15 documents


In [10]:
def naive_chunk(text, chunk_size=1200, overlap=100):
    """Baseline: dumb fixed-size character split, no table-awareness."""
    chunks, start = [], 0
    while start < len(text):
        chunk = text[start:start + chunk_size].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

def engineered_chunk(text, chunk_size=1800, overlap=250):
    """Engineered: split on section/table boundaries first, then size-limit within sections
    so a table is less likely to get cut in half mid-row."""
    sections = re.split(r"\n(?=#|\d+\.\s|Table\s|TABLE\s)", text)
    chunks = []
    for section in sections:
        section = section.strip()
        if not section:
            continue
        if len(section) <= chunk_size:
            chunks.append(section)
        else:
            start = 0
            while start < len(section):
                piece = section[start:start + chunk_size].strip()
                if piece:
                    chunks.append(piece)
                start += chunk_size - overlap
    return chunks

baseline_rows, engineered_rows = [], []
for _, row in docs_df.iterrows():
    for c in naive_chunk(row["text"]):
        baseline_rows.append({"text": c, "source_file": row["source_file"]})
    for c in engineered_chunk(row["text"]):
        engineered_rows.append({"text": c, "source_file": row["source_file"],
                                 "year": row["year"], "month": row["month"]})

baseline_chunks_df = pd.DataFrame(baseline_rows)
engineered_chunks_df = pd.DataFrame(engineered_rows)
print("Baseline chunks:", len(baseline_chunks_df))
print("Engineered chunks:", len(engineered_chunks_df))


Baseline chunks: 3832
Engineered chunks: 3184


## 5. Embedding + indexing both chunk sets with FAISS

In [11]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def build_faiss_index(chunks_df):
    embeddings = model.encode(chunks_df["text"].tolist(), show_progress_bar=True,
                               normalize_embeddings=True).astype("float32")
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    return index

baseline_index = build_faiss_index(baseline_chunks_df)
engineered_index = build_faiss_index(engineered_chunks_df)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/120 [00:00<?, ?it/s]

Batches:   0%|          | 0/100 [00:00<?, ?it/s]

## 6. Retrieval — plain top-K for baseline, metadata-filtered for engineered

In [12]:
MONTHS = {m: i+1 for i, m in enumerate(
    ["january","february","march","april","may","june","july",
     "august","september","october","november","december"])}
YEAR_IN_Q_RE = re.compile(r"\b(19|20)\d{2}\b")

def retrieve_baseline(question, k=K):
    q_emb = model.encode([question], normalize_embeddings=True).astype("float32")
    scores, ids = baseline_index.search(q_emb, k)
    return baseline_chunks_df.iloc[ids[0]].assign(score=scores[0]).to_dict("records")

def retrieve_engineered(question, k=K):
    q_lower = question.lower()
    year_match = YEAR_IN_Q_RE.search(question)
    month_match = next((num for name, num in MONTHS.items() if name in q_lower), None)

    pool = engineered_chunks_df
    if year_match:
        pool = pool[pool["year"] == int(year_match.group())]
    if month_match:
        pool = pool[pool["month"] == month_match]
    if pool.empty:
        pool = engineered_chunks_df

    q_emb = model.encode([question], normalize_embeddings=True).astype("float32")
    sub_texts = pool["text"].tolist()
    sub_embeddings = model.encode(sub_texts, normalize_embeddings=True).astype("float32")
    sub_index = faiss.IndexFlatIP(sub_embeddings.shape[1])
    sub_index.add(sub_embeddings)
    scores, ids = sub_index.search(q_emb, min(k, len(pool)))
    return pool.iloc[ids[0]].assign(score=scores[0]).to_dict("records")


## 7. Rule-based answer generation (no LLM API key needed)


In [13]:
NUM_RE = re.compile(r"[-+]?\d[\d,]*\.?\d*%?")

def extract_numbers(text):
    out = []
    for tok in NUM_RE.findall(text):
        is_pct = tok.endswith("%")
        try:
            val = float(tok.replace(",", "").replace("%", ""))
            out.append((val, is_pct, tok))
        except ValueError:
            continue
    return out

def clean_gold(gold):
    g = str(gold).replace(",", "").replace("$", "").replace("%", "").strip()
    try:
        return float(g)
    except ValueError:
        return None

def generate_answer(question, retrieved):
    """Pick the number, across all retrieved chunks, whose surrounding text best matches
    the question's key terms. Simple and transparent — a real generator would use an LLM,
    but this keeps everything runnable without an API key."""
    q_terms = set(re.findall(r"[a-z]{4,}", question.lower()))
    best_val, best_score = None, -1
    claims = []
    for r in retrieved:
        text = r["text"]
        for val, is_pct, raw in extract_numbers(text):
            window = text[max(0, text.find(raw) - 60): text.find(raw) + 60].lower()
            overlap = len(q_terms & set(re.findall(r"[a-z]{4,}", window)))
            claims.append(f"Found value {raw} near context: ...{window.strip()}...")
            if overlap > best_score:
                best_score = overlap
                best_val = raw
    return (best_val or ""), claims[:5]


## 8. Evaluation metrics

In [14]:
def source_from_chunk_id(row_source_file):
    return row_source_file

def hit_and_rank(gold_sources, retrieved):
    if not gold_sources:
        return None, None
    for rank, r in enumerate(retrieved, start=1):
        if r["source_file"] in gold_sources:
            return 1, rank
    return 0, None

def factual_accuracy_row(gold_answer, pred_answer, tolerance_pct=1.0):
    gold_num = clean_gold(gold_answer)
    pred_clean = str(pred_answer).replace(",", "").replace("$", "").replace("%", "").strip()
    try:
        pred_num = float(pred_clean)
    except ValueError:
        return int(str(gold_answer).strip().lower() in str(pred_answer).strip().lower())
    if gold_num is None:
        return 0
    if gold_num == 0:
        return int(pred_num == 0)
    return int(abs(gold_num - pred_num) / abs(gold_num) * 100 <= tolerance_pct)

def groundedness_and_halluc(pred_answer, retrieved):
    """A claim counts as grounded if the predicted number actually appears verbatim in the
    retrieved context (true here by construction, since we only ever extract numbers we saw —
    this metric mainly flags the empty-answer / not-found case)."""
    if not pred_answer:
        return 0.0, 0.0
    found = any(str(pred_answer) in r["text"] for r in retrieved)
    return (1.0, 0.0) if found else (0.0, 1.0)


## 9. Running baseline system over the filtered questions

In [15]:
baseline_rows = []
for _, row in tqdm(qa_recent.iterrows(), total=len(qa_recent)):
    question, gold = str(row["question"]), str(row["answer"])
    gold_sources = row["gold_source_files"]

    retrieved = retrieve_baseline(question)
    pred, claims = generate_answer(question, retrieved)
    hit, rank = hit_and_rank(gold_sources, retrieved)
    ground, halluc = groundedness_and_halluc(pred, retrieved)

    baseline_rows.append({
        "question": question, "gold_answer": gold, "predicted_answer": pred,
        "hit_at_5": hit, "rank": rank, "mrr": (1 / rank) if rank else 0,
        "groundedness": ground, "hallucination": halluc,
        "factual_accuracy": factual_accuracy_row(gold, pred),
    })

baseline_results = pd.DataFrame(baseline_rows)
baseline_results.to_csv(RESULTS_DIR / "baseline_results.csv", index=False)
baseline_results.head()


100%|██████████| 8/8 [00:00<00:00, 18.12it/s]


,question,gold_answer,predicted_answer,hit_at_5,rank,mrr,groundedness,hallucination,factual_accuracy
0,What percent did the Employment and General Re...,73,10,1,3.0,0.333333,1.0,0.0,0
1,How much does the U.S Treasury have invested i...,935851121560,-1,1,4.0,0.250000,1.0,0.0,0
2,Calculate the percentage difference relative t...,19.96%,3,1,4.0,0.250000,1.0,0.0,0
3,"At year-end for CY 2022, what was the total ou...","$23,918,635",-1,1,1.0,1.000000,1.0,0.0,0
4,What was the absolute QoQ percent change in to...,4.815,5.330,0,NaN,0.000000,1.0,0.0,0


## 10. Running engineered system over the filtered questions

In [16]:
engineered_rows = []
for _, row in tqdm(qa_recent.iterrows(), total=len(qa_recent)):
    question, gold = str(row["question"]), str(row["answer"])
    gold_sources = row["gold_source_files"]

    retrieved = retrieve_engineered(question)
    pred, claims = generate_answer(question, retrieved)
    hit, rank = hit_and_rank(gold_sources, retrieved)
    ground, halluc = groundedness_and_halluc(pred, retrieved)

    engineered_rows.append({
        "question": question, "gold_answer": gold, "predicted_answer": pred,
        "hit_at_5": hit, "rank": rank, "mrr": (1 / rank) if rank else 0,
        "groundedness": ground, "hallucination": halluc,
        "factual_accuracy": factual_accuracy_row(gold, pred),
    })

engineered_results = pd.DataFrame(engineered_rows)
engineered_results.to_csv(RESULTS_DIR / "engineered_results.csv", index=False)
engineered_results.head()


100%|██████████| 8/8 [31:49<00:00, 238.66s/it]


,question,gold_answer,predicted_answer,hit_at_5,rank,mrr,groundedness,hallucination,factual_accuracy
0,What percent did the Employment and General Re...,73,2025,0,NaN,0.0,1.0,0.0,0
1,How much does the U.S Treasury have invested i...,935851121560,0,0,NaN,0.0,1.0,0.0,0
2,Calculate the percentage difference relative t...,19.96%,0.9,1,2.0,0.5,1.0,0.0,0
3,"At year-end for CY 2022, what was the total ou...","$23,918,635",0,0,NaN,0.0,1.0,0.0,0
4,What was the absolute QoQ percent change in to...,4.815,62131,0,NaN,0.0,1.0,0.0,0


## 11. Scorecard

In [17]:
def summarize(df):
    valid_hits = df["hit_at_5"].dropna()
    return {
        "Hit Rate@5": valid_hits.mean() if len(valid_hits) else float("nan"),
        "MRR": df["mrr"].mean(),
        "Groundedness": df["groundedness"].mean(),
        "Factual Accuracy": df["factual_accuracy"].mean(),
        "Hallucination Rate": df["hallucination"].mean(),
    }

base_summary = summarize(baseline_results)
eng_summary = summarize(engineered_results)

scorecard = pd.DataFrame({
    "Metric": list(base_summary.keys()),
    "Baseline": list(base_summary.values()),
    "Engineered": list(eng_summary.values()),
})
scorecard.to_csv(RESULTS_DIR / "scorecard.csv", index=False)
scorecard


,Metric,Baseline,Engineered
0,Hit Rate@5,0.625000,0.1250
1,MRR,0.270833,0.0625
2,Groundedness,1.000000,1.0000
3,Factual Accuracy,0.000000,0.0000
4,Hallucination Rate,0.000000,0.0000


## 12. Zip everything for download

In [18]:
shutil.make_archive("/content/financial_rag_officeqa", "zip", PROJECT_DIR)
print("Zip created: /content/financial_rag_officeqa.zip ")


Zip created: /content/financial_rag_officeqa.zip 
